In [ ]:
%pip -q install openai pandas numpy scipy scikit-learn tqdm

In [ ]:
from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
import base64
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm
from openai import OpenAI

RUN_TRAIN_API_CALLS = False
RUN_DEV_API_CALLS = False
RUN_TEST_API_CALLS = False
CC_MODEL = 'gpt-5.4'
CC_REASONING_EFFORT = 'none'
CC_IMAGE_DETAIL = 'original'
CC_DEMONSTRATION_IMAGE_DETAIL = 'low'
REQUEST_SLEEP_SECONDS = 0.2
MAX_RETRIES = 3
TARGET_COLUMN = 'CRAI_CC'
SCORE_ANCHORS = {'0.00': 0.00, '0.25': 0.25, '0.50': 0.50, '0.75': 0.75, '1.00': 1.00}

drive.mount('/content/drive')
IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR = PROJECT_DIR / 'data'
EXPERIMENT_ROOT = PROJECT_DIR / 'cc_qatari_label_anchored_v3'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=userdata.get('openai'))

In [ ]:
VERIFIED_GENERATED_BASE_REMAP = {
    'train': {'img_017': 'img_018', 'img_018': 'img_019', 'img_019': 'img_020'}
}
UNRESOLVED_BASE_IDS = {'train': {'img_020'}}

def parse_image_base_id(instance_id):
    return re.sub(r'_v\d+$', '', str(instance_id))

def parse_caption_version(instance_id):
    return int(re.search(r'_v(\d+)$', str(instance_id)).group(1))

def find_existing_image(folder, stem):
    for extension in ['.png', '.jpg', '.jpeg', '.webp']:
        candidate = folder / f'{stem}{extension}'
        if candidate.exists():
            return str(candidate)

def resolve_generated_source_id(instance_id, split):
    base = parse_image_base_id(instance_id)
    source = VERIFIED_GENERATED_BASE_REMAP.get(split, {}).get(base, base)
    return f'{source}_v{parse_caption_version(instance_id)}'

def load_split(split, gold=False):
    folder = DATA_DIR / split
    frame = pd.read_csv(folder / 'captions.tsv', sep='\t')
    if gold:
        frame = frame.merge(pd.read_csv(folder / 'gold_human.tsv', sep='\t'), on='id')
    frame['id'] = frame['id'].astype(str)
    frame['split'] = split
    frame['base_id'] = frame['id'].map(parse_image_base_id)
    frame['caption_version'] = frame['id'].map(parse_caption_version)
    frame['caption_version_key'] = 'v' + frame['caption_version'].astype(str)
    frame['category'] = frame.get('category', 'unknown')
    frame['generated_source_id'] = frame['id'].map(lambda x: resolve_generated_source_id(x, split))
    frame['eligible_for_cc'] = ~frame['base_id'].isin(UNRESOLVED_BASE_IDS.get(split, set()))
    frame['ref_image_path'] = frame['base_id'].map(
        lambda x: find_existing_image(folder / 'imgs' / 'ref', x)
    )
    frame['generated_image_path'] = frame['generated_source_id'].map(
        lambda x: find_existing_image(folder / 'imgs' / 'generated', x)
    )
    return frame

def caption_text_from_row(row):
    return str(row['caption'])

def make_v1_caption_map(frame):
    v1 = frame[frame['caption_version'].eq(1)]
    return dict(zip(v1['base_id'], v1['caption']))

train_all_df = load_split('train', gold=True)
dev_all_df = load_split('dev', gold=True)
train_df = train_all_df[train_all_df['eligible_for_cc']].reset_index(drop=True)
dev_df = dev_all_df[dev_all_df['eligible_for_cc']].reset_index(drop=True)
test_all_df = load_split('test') if (DATA_DIR / 'test' / 'captions.tsv').exists() else None
test_df = test_all_df[test_all_df['eligible_for_cc']].reset_index(drop=True) if test_all_df is not None else None

V1_CAPTION_BY_SPLIT = {
    'train': make_v1_caption_map(train_all_df),
    'dev': make_v1_caption_map(dev_all_df),
}
if test_all_df is not None:
    V1_CAPTION_BY_SPLIT['test'] = make_v1_caption_map(test_all_df)

def get_v1_caption(row):
    return V1_CAPTION_BY_SPLIT[row['split']][row['base_id']]

In [ ]:
def image_to_data_url(path):
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def extract_caption(row):
    return str(row['caption'])

def parse_json_object(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):text.rfind('}') + 1])

def load_jsonl(path):
    if not path.exists():
        return []
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def append_jsonl(path, record):
    with path.open('a') as handle:
        handle.write(json.dumps(record) + '\n')

In [ ]:
CC_PROMPT_VERSION = 'qatari-label-anchored-cc-v3'

SCENE_CATEGORIES = {
    'people_practice', 'architecture_landmark', 'objects_market'
}

QATAR_CONTEXT_STATUSES = {'matched', 'partial', 'missing', 'conflicting'}

SCORE_BAND_KEYS = tuple(SCORE_ANCHORS)

CC_DEMONSTRATION_SPECS = [
    {
        'id': 'img_007_v3',
        'scene_category': 'people_practice',
        'qatar_context_status': 'conflicting',
        'brief_reason': (
            'The people and formal context do not preserve the intended Qatari scene.'
        ),
    },
    {
        'id': 'img_001_v5',
        'scene_category': 'people_practice',
        'qatar_context_status': 'partial',
        'brief_reason': (
            'The activity and coastal setting are partly coherent, but much of the '
            'intended Qatari context is lost.'
        ),
    },
    {
        'id': 'img_035_v3',
        'scene_category': 'objects_market',
        'qatar_context_status': 'partial',
        'brief_reason': (
            'The heritage-market organization is coherent but only partially preserves '
            'the intended Qatari context.'
        ),
    },
    {
        'id': 'img_007_v4',
        'scene_category': 'people_practice',
        'qatar_context_status': 'matched',
        'brief_reason': (
            'The portrait, attire, and formal setting form a coherent Qatari scene.'
        ),
    },
    {
        'id': 'img_029_v4',
        'scene_category': 'architecture_landmark',
        'qatar_context_status': 'conflicting',
        'brief_reason': (
            'The plausible coastal skyline does not preserve the intended Qatari '
            'landmark and spatial context.'
        ),
    },
    {
        'id': 'img_028_v4',
        'scene_category': 'architecture_landmark',
        'qatar_context_status': 'matched',
        'brief_reason': (
            'The Qatari architectural identity and its surrounding context are strongly '
            'preserved.'
        ),
    },
]

CC_DEMONSTRATION_IDS = [item['id'] for item in CC_DEMONSTRATION_SPECS]

CC_DEMONSTRATION_BASE_IDS = {
    parse_image_base_id(value) for value in CC_DEMONSTRATION_IDS
}

def score_to_band_probabilities(score: float) -> dict[str, float]:
    score = float(np.clip(score, 0.0, 1.0))
    keys = list(SCORE_ANCHORS)
    values = np.asarray([SCORE_ANCHORS[key] for key in keys], dtype=float)
    probabilities = np.zeros(len(values), dtype=float)
    if score <= values[0]:
        probabilities[0] = 1.0
    elif score >= values[-1]:
        probabilities[-1] = 1.0
    else:
        upper = int(np.searchsorted(values, score, side='right'))
        lower = upper - 1
        distance = values[upper] - values[lower]
        probabilities[upper] = (score - values[lower]) / distance
        probabilities[lower] = 1.0 - probabilities[upper]
    return {
        key: round(float(probability), 6)
        for key, probability in zip(keys, probabilities)
    }

def demonstration_row(instance_id: str) -> pd.Series:
    selected = train_all_df.loc[train_all_df['id'].eq(instance_id)]
    if len(selected) != 1:
        raise ValueError(f'Demonstration {instance_id!r} was not found uniquely')
    row = selected.iloc[0]
    if not bool(row['eligible_for_cc']):
        raise ValueError(f'Demonstration {instance_id!r} has unresolved mapping')
    return row

def demonstration_response(spec: dict) -> dict:
    row = demonstration_row(spec['id'])
    return {
        'scene_category': spec['scene_category'],
        'qatar_context_status': spec['qatar_context_status'],
        'score_probabilities': score_to_band_probabilities(row[TARGET_COLUMN]),
        'brief_reason': spec['brief_reason'],
    }

CC_DEMONSTRATION_RESPONSES = {
    spec['id']: demonstration_response(spec)
    for spec in CC_DEMONSTRATION_SPECS
}

train_judge_df = train_df.loc[
    ~train_df['base_id'].isin(CC_DEMONSTRATION_BASE_IDS)
].reset_index(drop=True)

STRUCTURED_DIRECT_CC_PROMPT = r"""
ROLE

You are a multimodal judge of Contextual Coherence (CC) for a Qatar-focused
cultural image dataset. Return valid JSON only.

TARGET CULTURE

Every reference image concerns Qatari people, places, practices, objects, or
cultural settings. Qatar is the only target culture. Do not treat "Gulf" or "Arab"
as alternative correct target cultures. A broader regional scene is a mismatch; it
may receive partial CC only when the complete candidate remains contextually coherent
and the current caption is broad. It cannot receive full Qatar-context credit.

INPUTS

You receive:
1. A Qatari reference image.
2. Its culturally explicit V1 source caption.
3. The current caption used to generate the candidate.
4. The generated candidate image.

TASK

Judge whether the visible elements in the candidate are placed together in a context
appropriate to the intended scene.

CC is not only caption-image similarity, not only correctness of isolated cultural
elements, and not only the quantity of Qatar-specific cues. Use the current caption
to determine requested content and relations. Use the reference image and V1 caption
to identify the intended Qatari scene, place, practice, and contextual organization.
Do not demand exact details that the current caption clearly omits unless their loss
makes the setting contextually wrong or generic.

CATEGORY-SPECIFIC RULES

Architecture or landmarks:
- Strongly weight the identity of the actual Qatari landmark, its surroundings, and
  spatial context.
- A coherent foreign, broader-regional, or generic landmark is not an appropriate
  replacement.
- Usually decide among strong match, partial resemblance, and clear mismatch.

People or cultural practices:
- Judge whether people, attire, activity, objects, and environment belong together.
- Correct Qatari attire alone does not guarantee high CC if the wider setting or
  activity is absent or inappropriate.
- A coherent but non-Qatari regional scene can receive partial, never full, credit
  when the current caption is broad.

Objects, crafts, or markets:
- Judge whether objects are placed, displayed, and used in an appropriate Qatari
  heritage, domestic, commercial, or activity setting.
- A generic bazaar or heritage setting receives only partial credit when its
  organization is plausible but Qatar-specific context is lost.

METRIC SEPARATION

- CEA evaluates correctness of individual cultural elements.
- CS evaluates cultural distinctiveness and Qatar-specificity.
- CC evaluates whether the complete scene and visible relations make contextual sense.
- A wrong national identity reduces CC and prevents a full score, but does not force
  zero when a broad current caption and the complete scene remain coherent.
- Ignore photorealism and aesthetics unless an artifact prevents interpretation.

SCORE BANDS

0.00: The intended setting is absent, generic when context is essential, visibly
wrong, or contradicted; important elements do not belong together.

0.25: Only weak contextual evidence is preserved; major setting, landmark, activity,
or cultural relations are incorrect.

0.50: The scene is meaningfully but incompletely coherent; it preserves part of the
intended context or presents a related setting while losing important Qatari context.

0.75: The scene is mostly contextually appropriate; primary setting and relations are
coherent, with a limited omission, substitution, or national-specificity mismatch.

1.00: The candidate strongly preserves the intended Qatari scene or setting, and all
important people, objects, activities, and spatial relations are appropriate.

OUTPUT

Return probabilities over the five score bands. Do not force certainty. All five
probabilities must be present, lie in [0,1], and sum approximately to 1.

{
  "scene_category": "people_practice | architecture_landmark | objects_market",
  "qatar_context_status": "matched | partial | missing | conflicting",
  "score_probabilities": {
    "0.00": 0.0,
    "0.25": 0.0,
    "0.50": 0.0,
    "0.75": 0.0,
    "1.00": 0.0
  },
  "brief_reason": "One sentence grounded in visible contextual evidence."
}

Return no markdown and no text outside the JSON object.
"""

In [ ]:
def structured_cc_user_content(
    row: pd.Series, image_detail: str, labelled_example: bool = False
) -> list[dict]:
    prefix = (
        'LABELLED TRAINING EXAMPLE' if labelled_example
        else 'UNLABELLED INSTANCE TO JUDGE'
    )
    text = f"""{prefix}
Instance ID: {row['id']}

V1 caption (Qatari source context):
{get_v1_caption(row)}

Current caption (v{row['caption_version']}):
{extract_caption(row)}

The first image is the Qatari reference. The second is the generated candidate.
Apply the category-specific CC rubric and return JSON only."""
    return [
        {'type': 'input_text', 'text': text},
        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
        {
            'type': 'input_image',
            'image_url': image_to_data_url(row['ref_image_path']),
            'detail': image_detail,
        },
        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
        {
            'type': 'input_image',
            'image_url': image_to_data_url(row['generated_image_path']),
            'detail': image_detail,
        },
    ]

def cc_demonstration_messages() -> list[dict]:
    messages = []
    for spec in CC_DEMONSTRATION_SPECS:
        row = demonstration_row(spec['id'])
        messages.append({
            'role': 'user',
            'content': structured_cc_user_content(
                row,
                image_detail=CC_DEMONSTRATION_IMAGE_DETAIL,
                labelled_example=True,
            ),
        })
        messages.append({
            'role': 'assistant',
            'content': json.dumps(
                CC_DEMONSTRATION_RESPONSES[spec['id']],
                ensure_ascii=False,
            ),
        })
    return messages
def call_structured_cc(row):
    messages = cc_demonstration_messages()
    messages.append({
        'role': 'user',
        'content': structured_cc_user_content(row, image_detail=CC_IMAGE_DETAIL),
    })
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=CC_MODEL,
                reasoning={'effort': CC_REASONING_EFFORT},
                input=[{'role': 'developer', 'content': STRUCTURED_DIRECT_CC_PROMPT}, *messages],
            )
            return parse_json_object(response.output_text)
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

def load_or_infer_structured_cc(frame, split, run_calls):
    path = CACHE_DIR / f'cc_structured_{split}.jsonl'
    cache = {record['instance_id']: record for record in load_jsonl(path)}
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f'CC {split}'):
        if row['id'] not in cache and run_calls:
            record = {'instance_id': row['id'], 'response': call_structured_cc(row)}
            cache[row['id']] = record
            append_jsonl(path, record)
            time.sleep(REQUEST_SLEEP_SECONDS)
    return pd.DataFrame([cache[x] for x in frame['id'] if x in cache])

train_cc_records = load_or_infer_structured_cc(train_judge_df, 'train', RUN_TRAIN_API_CALLS)
dev_cc_records = load_or_infer_structured_cc(dev_df, 'dev', RUN_DEV_API_CALLS)
test_cc_records = (
    load_or_infer_structured_cc(test_df, 'test', RUN_TEST_API_CALLS)
    if test_df is not None else pd.DataFrame()
)

In [ ]:
PROBABILITY_COLUMNS = {
    key: f'probability_{key.replace(".", "")}' for key in SCORE_BAND_KEYS
}

def cc_records_to_features(records, metadata):
    rows = []
    for record in records.to_dict('records'):
        response = record['response']
        probabilities = response['score_probabilities']
        values = np.array([probabilities[key] for key in SCORE_BAND_KEYS], dtype=float)
        positive = values[values > 0]
        row = {
            'id': record['instance_id'],
            'raw_prediction': float(response['raw_cc']),
            'judge_confidence': float(values.max()),
            'score_entropy': -float(np.sum(positive * np.log(positive))),
            'scene_category': response['scene_category'],
            'qatar_context_status': response['qatar_context_status'],
        }
        row.update({PROBABILITY_COLUMNS[key]: float(probabilities[key]) for key in SCORE_BAND_KEYS})
        rows.append(row)
    columns = ['id', 'base_id', 'caption_version', 'caption_version_key', 'category']
    return metadata[columns].merge(pd.DataFrame(rows), on='id') if rows else pd.DataFrame()

train_cc_features = cc_records_to_features(train_cc_records, train_judge_df)
dev_cc_features = cc_records_to_features(dev_cc_records, dev_df)
test_cc_features = cc_records_to_features(test_cc_records, test_df) if test_df is not None else pd.DataFrame()

In [ ]:
RAW_WEIGHT = 0.40

def version_prior(training, target):
    medians = training.groupby('caption_version')['CRAI_CC'].median()
    return target['caption_version'].map(medians).to_numpy(float)

if len(dev_cc_features) == len(dev_df):
    dev_prediction = RAW_WEIGHT * dev_cc_features['raw_prediction'].to_numpy(float)
    dev_prediction += (1 - RAW_WEIGHT) * version_prior(train_df, dev_df)
    dev_output = pd.DataFrame({'id': dev_df['id'], 'CRAI_CC': np.clip(dev_prediction, 0, 1)})
    evaluation = dev_df[['id', 'CRAI_CC']].merge(
        dev_output, on='id', suffixes=('_gold', '_prediction')
    )
    metrics = pd.DataFrame([{
        'spearman': spearmanr(evaluation['CRAI_CC_gold'], evaluation['CRAI_CC_prediction']).statistic,
        'mae': mean_absolute_error(evaluation['CRAI_CC_gold'], evaluation['CRAI_CC_prediction']),
    }])
    display(metrics.round(4))
    dev_output.to_csv(OUTPUT_DIR / 'cc_v3_dev_predictions.tsv', sep='\t', index=False)

if test_df is not None and len(test_cc_features) == len(test_df):
    test_prediction = RAW_WEIGHT * test_cc_features['raw_prediction'].to_numpy(float)
    test_prediction += (1 - RAW_WEIGHT) * version_prior(train_df, test_df)
    test_output = pd.DataFrame({
        'id': test_df['id'],
        'CRAI_CC': np.clip(test_prediction, 0, 1),
    })
    test_output.to_csv(OUTPUT_DIR / 'cc_v3_test_predictions.tsv', sep='\t', index=False)